# PyTorch-基本功能

## 教學目標

這份教學的目標是介紹 PyTorch，撰寫深度學習模型的函式庫。

## 適用對象

已經有基本的機器學習知識，且擁有 python、`numpy`、`matplotlib` 基礎的學生。

若沒有先學過 Python，建議先補充 Python 入門語法。

若沒有先學過 `numpy`，請參考 [numpy-基本功能](./numpy-基本功能.ipynb) 教學。

若沒有先學過 `matplotlib`，請參考 [matplotlib-資料視覺化](./matplotlib-資料視覺化.ipynb) 教學。

## 原教材的執行時間與測試環境

以下是 `original` 原教材的測試記錄，精簡版的執行時間會因內容與環境而不同。

本教學全部執行時間約為 4.376506090164185 秒。

|測試環境|名稱|
|-|-|
|主機板|X570 AORUS ELITE|
|處理器|AMD Ryzen 7 3700X 8-Core Processor|
|記憶體|Kingston KHX3200C16D4/16GX|
|硬碟|Seagate ST1000DM003-1ER1|
|顯示卡|GeForce RTX 3060|
|作業系統|Ubuntu 22.04 LTS|

本精簡版以已學過 vibe coding 的學生為對象，重點是讀懂資料、模型與訓練流程。

## 大綱

- 簡介與安裝
- Tensor 的 shape、資料型態與運算
- 深度學習流程、Dataset 與 DataLoader
- 神經網路、啟動函數、損失與最佳化
- 訓練、驗證、測試與模型保存
- 練習與延伸資源

## 簡介

根據 [PyTorch 官方網站](https://pytorch.org/)（穩定版 v2.0.1）：

> PyTorch is an open source machine learning framework that accelerates the path from research prototyping to production deployment.
> 
> PyTorch 是一個開源的機器學習框架，能夠幫助加速從研究原型到商業應用的轉換過程。

![PyTorch usage statistics](https://thegradient.pub/content/images/2019/10/ratio_medium-1.png)

根據[統計](https://thegradient.pub/state-of-ml-frameworks-2019-pytorch-dominates-research-tensorflow-dominates-industry/)，PyTorch 在各大機器學習會議使用率逐年上升，使用者選擇 PyTorch 的原因為：

- 簡單（Simplicity）
    - 使用 `python` 作為介面
    - 操作方法與 `numpy` 相似
- 好用的介面（Great API）
    - 沒有過多的抽象化
- 效能（Performance）

## 安裝

參考 [PyTorch 安裝頁面](https://pytorch.org/get-started/locally/)選擇作業系統與適合的 CPU／GPU 安裝方式。
本範例只需要 `torch`；已有 PyTorch 的環境可以直接執行。沒有支援 CUDA 的 GPU 也可使用 CPU。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD

print('PyTorch:', torch.__version__)
print('CUDA 可用:', torch.cuda.is_available())

## 張量：資料與運算

`torch.Tensor` 類似 NumPy 陣列，可保存多維資料；不同之處是它也支援 GPU 與自動微分。
ML 常用 `(樣本數, 特徵數)` 表示輸入，應先檢查 `shape`、`dtype` 與 `device`。
`reshape` 可以改變形狀，但元素總數必須相同；`view` 回傳與原張量共用資料的視圖，
`reshape` 則可能回傳視圖或複本，不能假設修改其中一個一定會改變另一個。

In [ ]:
X = torch.tensor([[1., 2.], [3., 4.], [5., 6.]])
y = torch.tensor([[1.], [2.], [3.]])
print('X:', X.shape, X.dtype, X.device, 'y:', y.shape)
print('第一個特徵：', X[:, 0])
print('保留單一特徵維度：', X[:, :1].shape)
print('每個特徵的平均：', X.mean(dim=0))
print('重新排列：', X.reshape(2, 3))

張量可逐元素運算；`@` 是矩陣乘法。形狀相容時，較小的張量可以透過 broadcasting
套用到較大的張量。以下 `(3, 2) @ (2, 1)` 得到 `(3, 1)`，代表每筆樣本的一個輸出。

In [ ]:
weight = torch.tensor([[0.5], [-1.]])
bias = torch.tensor([1.])
print('逐元素相乘的 shape：', (X * X).shape)
print('矩陣乘法與 bias：', X @ weight + bias)
print('預測 shape：', (X @ weight + bias).shape)

## 使用 GPU 運算

運算用到的模型和張量需位於相同裝置。`torch.cuda.is_available()` 檢查 CUDA 是否可用，
`.to(device)` 將張量或模型移到選定裝置；沒有 CUDA 時使用 CPU。
Tensor 與 NumPy 可以互相轉換：此處使用 `.cpu().numpy()` 前先將不需梯度的結果移回 CPU。

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
X_device = X.to(device)
print('使用裝置：', device, '張量裝置：', X_device.device)
print('轉成 NumPy：', X_device.cpu().numpy())

## 深度學習

![Deep Learning](https://miro.medium.com/max/1000/1*51D0MqtqHu3h2vTE5oJ-7g.png)
![Deep Learning vs. Machine Learning](https://learn.microsoft.com/zh-tw/azure/machine-learning/media/concept-deep-learning-vs-machine-learning/ai-vs-machine-learning-vs-deep-learning.png)
### 深度學習要幹嘛？
上課教的 SVM, Bayes 等等的模型算是 （傳統）Machine Learning 的範疇，比較偏向建立於統計的假設的數學統計模型。而深度學習則是機器學習內的一個子集合（subset），他是以類神經網路來模擬人類大腦的神經元之間的互動。




![](https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/03-pytorch-computer-vision-workflow.png)

使用 `torch` 進行深度學習主要包含以下步驟：
1. 將資料轉換成 `torch.Tensor`，使用 Dataset 和 Dataloader 包裝管理資料。
2. 使用 `torch.nn` 建立深度學習模型架構。
3. 從 `torch.optim` 選擇最佳化工具。
4. 選擇目標函數。
5. 訓練深度學習模型（train）。
6. 測試深度學習模型（test, inference）。
以下範例：使用兩層全連接層做 Polynomial Regression 任務。

### 資料集與批次

範例目標是學習 $y = 2x^2 + 3x + 17$。`Dataset` 管理樣本與答案，
`DataLoader` 將它們組成 mini-batch；`shuffle=True` 在每次迭代時打亂樣本順序，
同一筆樣本的特徵與答案仍會一起取出。

一個 epoch 是完整走過訓練資料一次；一個 batch 通常對應一次參數更新。
如果資料筆數不能被 batch size 整除，預設最後一批會較小。
這裡每筆 `x` 與 `y` 都保留一個特徵維度，讓批次輸入與模型輸出均為 `(batch_size, 1)`。

較小的 batch 通常每個 epoch 更新較多次，梯度波動也較大；較大的 batch 通常需要更多記憶體。不同大小對訓練速度與效果的影響，仍需依任務和硬體比較。

In [ ]:
torch.manual_seed(42)

class MyDataset(Dataset):
    def __init__(self, size):
        self.x = torch.rand(size, 1)
        self.y = 2 * self.x ** 2 + 3 * self.x + 17

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        return self.x[index], self.y[index]

sample_dataset = MyDataset(10)
sample_loader = DataLoader(sample_dataset, batch_size=3, shuffle=False)
for batch_x, batch_y in sample_loader:
    print('X:', batch_x.shape, 'y:', batch_y.shape)

### 小練習 7：理解批次與資料對應

```python
dataset = MyDataset(10)
loader = DataLoader(dataset, batch_size=3, shuffle=False)
for batch_x, batch_y in loader:
    print(batch_x.shape, batch_y.shape)
```

請解釋程式會印出幾組 shape、最後一組為何不同，以及若將 `shuffle` 改為 `True`，
樣本順序和每筆樣本與答案的對應關係會如何改變。請在下方空白儲存格以 `#` 註解作答。

<details>
<summary>參考答案</summary>

總共印出 4 組；前 3 組的 `batch_x` 和 `batch_y` 均為 `(3, 1)`，最後一組均為 `(1, 1)`。
因為 10 筆資料分成每批 3 筆後剩下 1 筆，而預設會保留最後一批。

`shuffle=True` 會打亂抽取樣本的順序；每筆樣本與其答案仍成對取出。批次數量與大小不變，
但各批次包含的樣本可能不同。

</details>

### 建立模型

`torch.nn` 提供常用的神經層，例如 `Linear`、`Embedding`、`Conv1d`／`Conv2d`／`Conv3d`、
`RNN`、`LSTM` 與 `Transformer`。選擇哪一種取決於資料與任務。
透過繼承 `nn.Module` 建立模型：`__init__` 定義層，`forward` 定義資料如何經過這些層。

線性層以 $z = Wx + b$ 轉換特徵。本例使用兩個線性層及 ReLU：
`(batch_size, 1) → (batch_size, 10) → (batch_size, 1)`。

### 啟動函數（Activation Functions）

若模型需要啟動函數，可以使用 `torch.nn.functional` 中事先定義好的啟動函數：
常見的啟動函數包含：

|啟動函數|名稱|定義|數值範圍|
|-|-|-|-|
|`torch.nn.functional.relu`|ReLU|$$f(x_i) = \max(0, x_i)$$|$$[0, \infty)$$|
|`torch.nn.functional.softmax`|Softmax|$$f(x_i) = \frac{e^{x_i}}{\sum_{j = 0}^n e^{x_j}}$$|$$[0, 1]$$|
|`torch.nn.functional.sigmoid`|Sigmoid|$$f(x_i) = \frac{1}{1 + e^{-x_i}}$$|$$[0, 1]$$|
|`torch.nn.functional.tanh`|Hyperbolic Tangent|$$f(x_i) = \frac{e^{x_i} - e^{-x_i}}{e^{x_i} + e^{-x_i}}$$|$$[-1, 1]$$|



ReLU 將負值變為 0，並保留非負值；在多個線性層之間加入非線性，能讓網路表達更複雜的關係。
Softmax 將一組分數轉成總和為 1 的機率分布；分類的損失函數與輸出格式應配合任務選擇。
Sigmoid 和 tanh 在輸入飽和時可能產生很小的梯度。

In [ ]:
class MyModel(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim):
        super().__init__()
        self.layer1 = nn.Linear(in_dim, hid_dim)
        self.layer2 = nn.Linear(hid_dim, out_dim)

    def forward(self, batch_x):
        hidden = F.relu(self.layer1(batch_x))
        return self.layer2(hidden)

model = MyModel(in_dim=1, hid_dim=10, out_dim=1).to(device)
batch_x, batch_y = next(iter(sample_loader))
batch_x, batch_y = batch_x.to(device), batch_y.to(device)
pred_y = model(batch_x)
print('輸入、答案、預測：', batch_x.shape, batch_y.shape, pred_y.shape)

### 目標函數（Objective Functions）

使用 `torch.nn` 中事先定義好的目標函數進行模型最佳化，計算模型預測結果與標記訓練資料的誤差值，並透過向後傳播（Back Propagation）演算法取得相對於誤差值的梯度（Gradient）。
- 基本上就是去計算每個參數的偏微分。
- Backpropagation 用鏈鎖律計算梯度；gradient descent 則使用這些梯度更新參數。
- 使用了 Chain Rule 的概念。計算每個參數內對 cost function 的偏微分。這個 cost function 就是「正確答案與預測答案的距離」，更適當的名字是 loss function。
- 對一個 weight matrix $W \in R^{a \times b}$ 內，$W_{ij}$ 是 $l$層$i$ 神經元與 $l-1$ 層 $j$ 神經元之間的權重。
    - ![](https://i.imgur.com/fk5HCm4.png)
- 想更新這個權重值，就要計算 $\frac{\partial C}{\partial W_{ij}}$，其中 $C$ 是 loss function。使用 Chain Rule，可以拆解成 $\frac{\partial C}{\partial W_{ij}} =  \frac{\partial C}{\partial Z_{i}} \frac{\partial Z_{i}}{\partial W_{ij}}$，$Z_i$ 是 $i$ 所在的神經元的輸入值，$a^i$ 是 $i$ 神經元的輸出值。（下圖上標意義解釋：上標 $l$ 是你的層數（你現在更新的是哪一層的權重矩陣？實際上每一層都需要算），$r$ 是第$r$組資料）。所以這兩個 terms 分別要怎麼計算？請去看下面第一個影片。
    - ![](https://i.imgur.com/3EihDh7.png)
- Resources (a lot of maths): 
    - [DNN Back propagation](http://speech.ee.ntu.edu.tw/~tlkagk/courses/MLDS_2015_2/Lecture/DNN%20backprop.ecm.mp4/index.html)
    - [Computational Graph 的角度去看 Back propagation](https://www.youtube.com/watch?v=-yhm3WdGFok)

```python
# 匯入神經網路模型
import torch.nn as nn             

# 創造均方誤差計算工具
criterion = nn.MSELoss()          

# 計算 batch_x 得到 pred_y
pred_y = model(batch_x)        
# 計算 pred_y 與 batch_y 的均方誤差
loss = criterion(pred_y, batch_y) 

# 使用向後傳播計算梯度
loss.backward()                   
```

常見的目標函數包含：

|目標函數|名稱|
|-|-|
|`torch.nn.MSELoss`|均方誤差（Mean Square Error）|
|`torch.nn.CrossEntropyLoss`|交叉熵（Cross Entropy）|
|`torch.nn.BCELoss`|二元交叉熵（Binary Cross Entropy）|
|`torch.nn.NLLLoss`|負對數似然（Negative Log Likelihood）|

## MSELoss 
- [PyTorch Documentation](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)
<img src="https://i.imgur.com/dqKaK25.png"  width="400">

### 最佳化（Optimization）

`loss.backward()` 依計算圖與鏈鎖律計算梯度；`optimizer.step()` 再使用梯度更新參數。
SGD 的更新概念為 $\theta_t = \theta_{t-1} - \mathrm{lr}\,\nabla_\theta L$，
學習率 `lr` 控制更新幅度。最佳化器需要知道要更新哪些參數，可用 `model.parameters()` 提供。

PyTorch 的梯度會累積，因此每次計算新的 batch 梯度前使用 `optimizer.zero_grad()`。
batch size、學習率和模型大小都可能影響訓練結果；實際效果需透過驗證資料評估。
可參考原教材的 [SGD 文件](https://pytorch.org/docs/stable/optim.html#torch.optim.SGD)
與[梯度下降課程](https://www.youtube.com/watch?v=yKKNr-QKz2Q&list=PLJV_el3uVTsPy9oCRY30oBPNLCo89yu49&index=6)。

### 訓練、驗證與測試

流程是取出一批資料 → 預測 → 計算損失 → 清空舊梯度 → 反向傳播 → 更新參數。
本例的輸出與答案皆為 `(batch_size, 1)`，避免不相容的形狀影響損失計算。

每個 epoch 後在驗證集評估，協助觀察訓練；調整設計後再用獨立測試集評估。
評估時用 `torch.no_grad()` 關閉梯度紀錄；以下平均損失依各批實際筆數加權，
因此最後一批較小時也能得到逐樣本的平均 MSE。

In [ ]:
train_dataset = MyDataset(160)
valid_dataset = MyDataset(40)
test_dataset = MyDataset(40)
batch_size, n_epoch, hid_dim = 32, 60, 10
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model = MyModel(in_dim=1, hid_dim=hid_dim, out_dim=1).to(device)
criterion = nn.MSELoss()
optimizer = SGD(model.parameters(), lr=0.01)

def evaluate(loader):
    total_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            loss = criterion(model(batch_x), batch_y)
            total_loss += float(loss) * len(batch_x)
    return total_loss / len(loader.dataset)

for epoch in range(n_epoch):
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        pred_y = model(batch_x)
        loss = criterion(pred_y, batch_y)
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f'epoch {epoch + 1}: train MSE={evaluate(train_loader):.4f}, '
              f'validation MSE={evaluate(valid_loader):.4f}')

print(f'test MSE: {evaluate(test_loader):.4f}')

### 小練習 8：訓練與驗證的梯度

```python
for batch_x, batch_y in train_loader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
    optimizer.zero_grad()
    prediction = model(batch_x)
    loss = criterion(prediction, batch_y)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    for batch_x, batch_y in valid_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        validation_loss = criterion(model(batch_x), batch_y)
        print(validation_loss)
```

請說明每個訓練 batch 的梯度如何用來更新模型、移除 `zero_grad()` 會發生什麼事，
以及驗證區塊是否會更新模型參數、`no_grad()` 的作用是什麼。請在下方空白儲存格以 `#` 註解作答。

<details>
<summary>參考答案</summary>

每批先清空舊梯度，根據預測與答案計算損失，再由 `backward()` 計算梯度，最後由
`step()` 使用梯度更新參數。若移除 `zero_grad()`，新梯度會累積到先前的梯度，
後續更新會受到前面批次影響；累積梯度可以刻意使用，但必須明確設計。

驗證區塊只計算與列印損失，沒有呼叫 `backward()` 或 `step()`，因此不會更新參數。
`no_grad()` 停止記錄梯度運算以節省資源；即使拿掉它，只要仍不執行反向傳播與參數更新，
此區塊也不會更新參數。

</details>

### 測試、視覺化與模型保存

`torch.no_grad()` 適合評估或推論，不會自行更新模型參數。
檢查表現時，除了 MSE，也可像原教材一樣繪製測試資料的真實值與預測值，觀察哪些區域誤差較大。


以下延續原教材的視覺化範例，可在完成訓練後執行；需要 `matplotlib`：

```python
import matplotlib.pyplot as plt

with torch.no_grad():
    test_x = test_dataset.x.to(device)
    test_pred = model(test_x).cpu()
plt.scatter(test_dataset.x[:, 0], test_dataset.y[:, 0], label='Actual')
plt.scatter(test_dataset.x[:, 0], test_pred[:, 0], label='Predicted')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()
```

要保存模型時，可用 `torch.save(model.state_dict(), 路徑)`；載入時先建立相同架構，
再呼叫 `model.load_state_dict(torch.load(路徑))`。以下是示意，不會自動寫入檔案：

```python
# torch.save(model.state_dict(), 'model.ckpt')
# model.load_state_dict(torch.load('model.ckpt'))
```

## 練習

### 練習 1：調整超參數

請試著更改前述範例中的超參數讓模型表現變好：

- 增加訓練次數 `n_epoch`
- 增大單一訓練資料次數 `batch_size`
- 增大隱藏層的維度 `hid_dim`
- 更改啟動函數 `F.relu`

### 練習 2：加深模型

請試著更改前述範例中的模型深度讓模型表現變好：

- 增加 1 個或多個 `nn.Linear`

## 簡單的機器學習任務練習
- MNIST 資料集：手寫數字圖像分類任務 (Image Classification)
  - ![](https://thumbs.gfycat.com/AdorableJoyfulLemming-max-1mb.gif)
  - MNIST 一筆 data $\in \mathbb{R}^{784}$ （784 維的 feature vector）
  - [datahacker/cnn/#005 PyTorch - Convolutional Neural Network on MNIST Handwritten Digit Recognition in PyTorch 1.3.ipynb](https://github.com/maticvl/dataHacker/blob/master/CNN/%23005%20PyTorch%20-%20Convolutional%20Neural%20Network%20on%20MNIST%20Handwritten%20Digit%20Recognition%20in%20PyTorch%201.3.ipynb)
- Kaggle Titanic Survival Prediction (Feature Classification)
    - [Example Notebook: Introduction to Pytorch (a very gentle start)](https://www.kaggle.com/code/frtgnn/introduction-to-pytorch-a-very-gentle-start)
- Real-and-Fake Text Classification (Text Classifcation)
    - [A step-by-step guide: LSTM Text Classification Using Pytorch](https://towardsdatascience.com/lstm-text-classification-using-pytorch-2c6c657f8fc0)
- 分類問題是最簡單也是最核心的，所有任務（文字生成 text-generation，文字翻譯 translation，問答 QA，序列標注 sequence labeling 等等）全都是分類問題的變形而已，只是因為資料形式不同，這些任務的資料前處理與模型預測後處理會比分類問題更加複雜和不直覺。建議從最直觀的分類問題下手練習。
- 例子都是提供 PyTorch 架構的範例。

### 推薦深度學習或自然語言處理學習資源
  - 比較輕鬆入門深度學習的方式(老師很好笑，數學很硬）：
    - [Hung-Yi Lee 's NTU ML Course Website](https://speech.ee.ntu.edu.tw/~hylee/ml/2022-spring.php)
      - 如果想要更加熟悉，可以寫裡面的作業（大概寫 Regression, Classifcation, CNN, Self-attention, Transformer, BERT 就能夠 cover 這堂課需要的所有技能，比較有趣的作業可以寫看看 Explainable AI）。
    - [Hung Yi Lee's Youtube Channel](https://www.youtube.com/@HungyiLeeNTU)
  - 史丹佛的教科書/NTU 資訊所 NLP 課程用書：[Stanford Textbook: Speech & language Processing](https://web.stanford.edu/~jurafsky/slp3/) 
這本書寫很仔細，從 sequence models 和 word embeddings 開始講，後面的章節是分成多個不同 NLP 任務講解。想要知道一些 NLP 的基礎，詳讀以下章節應該可以收穫良多：
    - 6:	Vector Semantics and Embeddings	6: Vector Semantics	[new in this edition]
    - 7:	Neural Networks and Neural Language Models	7: Neural Networks [new in this edition]
    - 9:	RNNs and LSTMs		[new in this edition]
    - 10:	Transformers and Pretrained Language Models
  - Andrew Ng's Machine Learning Resources
    - [Sequence Models](https://www.coursera.org/learn/nlp-sequence-models)
  - [PyTorch Introduction by NTU EE](https://colab.research.google.com/drive/1Xed5YSpLsLfkn66OhhyNzr05VE89enng#scrollTo=jifMOIcNMTh5)